In [14]:
# Data Handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Machine Learning (for later use)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

print("All libraries imported successfully 🚀")

All libraries imported successfully 🚀


In [ ]:
import streamlit as st
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Load the trained model
model = joblib.load('delay_model.pkl')

# Load the dataset for insights and dashboard
df = pd.read_csv('airline_operations.csv')
df.fillna(0, inplace=True)
df['delay_rate'] = df['arr_del15'] / df['arr_flights']
df['high_delay'] = (df['delay_rate'] > 0.15).astype(int)

# Set page config
st.set_page_config(page_title="Airline Delay Predictor", layout="wide")

# Sidebar for navigation
st.sidebar.title("Navigation")
page = st.sidebar.radio("Go to", ["Prediction", "Airport Insights", "Dashboard"])

if page == "Prediction":
    st.title("Flight Delay Risk Prediction")
    st.markdown("Enter the flight details to predict delay risk.")
    
    # Input fields
    arr_flights = st.number_input("Arriving Flights", min_value=0, value=100)
    carrier_ct = st.number_input("Carrier Delays", min_value=0, value=5)
    weather_ct = st.number_input("Weather Delays", min_value=0, value=2)
    nas_ct = st.number_input("NAS Delays", min_value=0, value=3)
    late_aircraft_ct = st.number_input("Late Aircraft Delays", min_value=0, value=1)
    
    if st.button("Predict"):
        input_data = {
            'arr_flights': arr_flights,
            'carrier_ct': carrier_ct,
            'weather_ct': weather_ct,
            'nas_ct': nas_ct,
            'late_aircraft_ct': late_aircraft_ct
        }
        prediction = model.predict(pd.DataFrame([input_data]))[0]
        risk = "High Delay Risk" if prediction == 1 else "Low Delay Risk"
        st.success(f"Prediction: {risk}")

elif page == "Airport Insights":
    st.title("Airport Insights")
    st.markdown("Select an airport to view insights.")
    
    airports = df['airport_name'].unique()
    selected_airport = st.selectbox("Select Airport", airports)
    
    airport_data = df[df['airport_name'] == selected_airport]
    avg_delay_rate = airport_data['delay_rate'].mean()
    total_flights = airport_data['arr_flights'].sum()
    total_delays = airport_data['arr_del15'].sum()
    
    st.write(f"**Average Delay Rate:** {avg_delay_rate:.2%}")
    st.write(f"**Total Flights:** {total_flights}")
    st.write(f"**Total Delays:** {total_delays}")
    
    # Simple chart
    fig, ax = plt.subplots()
    airport_data.groupby('month')['delay_rate'].mean().plot(kind='bar', ax=ax)
    ax.set_title(f"Monthly Delay Rate for {selected_airport}")
    ax.set_ylabel("Delay Rate")
    st.pyplot(fig)

elif page == "Dashboard":
    st.title("Dashboard")
    st.markdown("Overview of airline operations.")
    
    # Basic charts
    col1, col2 = st.columns(2)
    
    with col1:
        st.subheader("Delay Rate by Carrier")
        carrier_delay = df.groupby('carrier_name')['delay_rate'].mean().sort_values(ascending=False).head(10)
        fig, ax = plt.subplots()
        carrier_delay.plot(kind='bar', ax=ax)
        ax.set_ylabel("Average Delay Rate")
        st.pyplot(fig)
    
    with col2:
        st.subheader("High Delay Distribution")
        fig, ax = plt.subplots()
        df['high_delay'].value_counts().plot(kind='pie', ax=ax, autopct='%1.1f%%', labels=['Low', 'High'])
        ax.set_title("High Delay Risk Distribution")
        st.pyplot(fig)
    
    # Additional chart
    st.subheader("Delay Rate Over Years")
    fig, ax = plt.subplots()
    df.groupby('year')['delay_rate'].mean().plot(ax=ax)
    ax.set_ylabel("Average Delay Rate")
    st.pyplot(fig)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import joblib

# Step 1: Data Preprocessing
def load_and_preprocess_data(file_path):
    # Load dataset
    df = pd.read_csv(file_path)
    
    # Handle missing values by replacing with 0
    df.fillna(0, inplace=True)
    
    # Create new features
    df['delay_rate'] = df['arr_del15'] / df['arr_flights']
    df['cancel_rate'] = df['arr_cancelled'] / df['arr_flights']
    
    # Create binary target column
    df['high_delay'] = (df['delay_rate'] > 0.15).astype(int)
    
    return df

# Step 2: Feature Selection
def select_features(df):
    features = ['arr_flights', 'carrier_ct', 'weather_ct', 'nas_ct', 'late_aircraft_ct']
    target = 'high_delay'
    
    X = df[features]
    y = df[target]
    
    return X, y

# Step 3: Train-Test Split
def split_data(X, y, test_size=0.2, random_state=42):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    return X_train, X_test, y_train, y_test

# Step 4: Model Training
def train_model(X_train, y_train):
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    return model

# Optional: Hyperparameter Tuning
def tune_hyperparameters(X_train, y_train):
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }
    grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='accuracy')
    grid_search.fit(X_train, y_train)
    return grid_search.best_estimator_

# Step 5: Model Evaluation
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    print("Accuracy Score:", accuracy_score(y_test, y_pred))
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Step 6: Feature Importance
def plot_feature_importance(model, features):
    importances = model.feature_importances_
    plt.figure(figsize=(10, 6))
    plt.barh(features, importances)
    plt.xlabel('Importance')
    plt.title('Feature Importance')
    plt.show()

# Step 7: Save Model
def save_model(model, filename='delay_model.pkl'):
    joblib.dump(model, filename)
    print(f"Model saved as {filename}")

# Step 8: Prediction Function
def predict_delay_risk(model, input_data):
    # input_data should be a dict with keys: arr_flights, carrier_ct, weather_ct, nas_ct, late_aircraft_ct
    df_input = pd.DataFrame([input_data])
    prediction = model.predict(df_input)[0]
    risk = "High" if prediction == 1 else "Low"
    return risk

# Main Pipeline
if __name__ == "__main__":
    # Assuming the dataset is 'airline_operations.csv' in the same directory
    file_path = 'airline_operations.csv'
    
    # Preprocess data
    df = load_and_preprocess_data(file_path)
    
    # Select features
    X, y = select_features(df)
    
    # Split data
    X_train, X_test, y_train, y_test = split_data(X, y)
    
    # Train model (or tune hyperparameters)
    # model = train_model(X_train, y_train)  # Use this for default params
    model = tune_hyperparameters(X_train, y_train)  # Use this for tuned params
    
    # Evaluate model
    evaluate_model(model, X_test, y_test)
    
    # Plot feature importance
    plot_feature_importance(model, X.columns)
    
    # Save model
    save_model(model)
    
    # Example prediction
    sample_input = {
        'arr_flights': 100,
        'carrier_ct': 5,
        'weather_ct': 2,
        'nas_ct': 3,
        'late_aircraft_ct': 1
    }
    risk = predict_delay_risk(model, sample_input)
    print(f"Predicted Delay Risk: {risk}")

In [ ]:
## Project Goals

- Analyze airline operational performance.
- Identify major delay causes.
- Study cancellation trends.
- Analyze airport and carrier-level performance.
- Identify seasonal delay patterns.

## KPIs

- Total Arrival Flights
- Total Delayed Flights (>15 min)
- Delay Rate (%)
- Cancellation Rate (%)
- Average Delay Minutes
- Delay Cause Contribution


In [16]:
import pandas as pd

df = pd.read_csv("airline_operations.csv")


df.head()


,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2022,5,9E,Endeavor Air Inc.,ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",136.0,7.0,5.95,0.00,...,0.0,1.00,0.0,0.0,255.0,222.0,0.0,4.0,0.0,29.0
1,2022,5,9E,Endeavor Air Inc.,ABY,"Albany, GA: Southwest Georgia Regional",91.0,16.0,7.38,0.00,...,0.0,6.09,0.0,0.0,884.0,351.0,0.0,81.0,0.0,452.0
2,2022,5,9E,Endeavor Air Inc.,ACK,"Nantucket, MA: Nantucket Memorial",19.0,2.0,0.13,0.00,...,0.0,0.88,1.0,0.0,138.0,4.0,0.0,106.0,0.0,28.0
3,2022,5,9E,Endeavor Air Inc.,AEX,"Alexandria, LA: Alexandria International",88.0,14.0,7.26,0.76,...,0.0,1.64,0.0,0.0,947.0,585.0,35.0,125.0,0.0,202.0
4,2022,5,9E,Endeavor Air Inc.,AGS,"Augusta, GA: Augusta Regional at Bush Field",181.0,19.0,13.84,0.00,...,0.0,2.09,0.0,0.0,808.0,662.0,0.0,87.0,0.0,59.0


In [20]:
df.info()
df.isnull().sum()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318017 entries, 0 to 318016
Data columns (total 21 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   year                 318017 non-null  int64  
 1   month                318017 non-null  int64  
 2   carrier              318013 non-null  object 
 3   carrier_name         318013 non-null  object 
 4   airport              318014 non-null  object 
 5   airport_name         318017 non-null  object 
 6   arr_flights          317524 non-null  float64
 7   arr_del15            317285 non-null  float64
 8   carrier_ct           317525 non-null  float64
 9   weather_ct           317523 non-null  float64
 10  nas_ct               317529 non-null  float64
 11  security_ct          317529 non-null  float64
 12  late_aircraft_ct     317529 non-null  float64
 13  arr_cancelled        317529 non-null  float64
 14  arr_diverted         317527 non-null  float64
 15  arr_delay        

,year,month,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
count,318017.000000,318017.000000,317524.000000,317285.000000,317525.000000,317523.000000,317529.000000,317529.000000,317529.000000,317529.000000,317527.000000,317523.000000,317525.000000,317529.000000,317529.000000,317527.000000,317529.000000
mean,2012.450957,6.497844,381.766670,72.905076,21.073149,2.616407,24.005228,0.179037,24.975734,7.207257,0.867674,4209.989113,1286.577224,220.567542,1099.516422,7.214845,1596.062993
std,5.678296,3.459423,1027.156722,198.936754,47.671580,9.968640,85.113757,0.844834,75.275223,37.216301,3.915772,12519.021012,3515.417309,861.521440,4636.475908,38.854685,4924.950687
min,2003.000000,1.000000,1.000000,0.000000,0.000000,0.000000,-0.010000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-19.000000,0.000000,0.000000
25%,2007.000000,3.000000,59.000000,9.000000,3.000000,0.000000,1.680000,0.000000,1.640000,0.000000,0.000000,436.000000,148.000000,0.000000,56.000000,0.000000,79.000000
50%,2012.000000,6.000000,120.000000,23.000000,8.190000,0.580000,5.490000,0.000000,5.860000,1.000000,0.000000,1201.000000,437.000000,25.000000,203.000000,0.000000,351.000000
75%,2018.000000,10.000000,273.000000,56.000000,19.690000,2.000000,15.350000,0.000000,17.040000,4.000000,1.000000,3080.000000,1100.000000,159.000000,602.000000,0.000000,1110.000000
max,2022.000000,12.000000,21977.000000,6377.000000,1792.070000,717.940000,4091.270000,80.560000,1885.470000,4951.000000,256.000000,433687.000000,196944.000000,57707.000000,238440.000000,3760.000000,148181.000000


In [18]:
df.shape


(318017, 21)

In [23]:
df.memory_usage(deep=True)
df['year'] = df['year'].astype('int16')
df['month'] = df['month'].astype('int8')



In [26]:
df.isnull().sum()


year                   0
month                  0
carrier                0
carrier_name           0
airport                0
airport_name           0
arr_flights            0
arr_del15              0
carrier_ct             0
weather_ct             0
nas_ct                 0
security_ct            0
late_aircraft_ct       0
arr_cancelled          0
arr_diverted           0
arr_delay              0
carrier_delay          0
weather_delay          0
nas_delay              0
security_delay         0
late_aircraft_delay    0
dtype: int64

In [25]:
df.fillna(0, inplace=True)


In [29]:
df['delay_rate'] = df['arr_del15'] / df['arr_flights']
df['cancel_rate'] = df['arr_cancelled'] / df['arr_flights']
df['weather_delay_pct'] = df['weather_ct'] / df['arr_del15']
df['carrier_delay_pct'] = df['carrier_ct'] / df['arr_del15']




In [31]:
df['date'] = pd.to_datetime(df[['year','month']].assign(day=1))
df.head()

,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,delay_rate,cancel_rate,weather_delay_pct,carrier_delay_pct,date
0,2022,5,9E,Endeavor Air Inc.,ABE,"Allentown/Bethlehem/Easton, PA: Lehigh Valley ...",136.0,7.0,5.95,0.00,...,222.0,0.0,4.0,0.0,29.0,0.051471,0.000000,0.000000,0.850000,2022-05-01
1,2022,5,9E,Endeavor Air Inc.,ABY,"Albany, GA: Southwest Georgia Regional",91.0,16.0,7.38,0.00,...,351.0,0.0,81.0,0.0,452.0,0.175824,0.000000,0.000000,0.461250,2022-05-01
2,2022,5,9E,Endeavor Air Inc.,ACK,"Nantucket, MA: Nantucket Memorial",19.0,2.0,0.13,0.00,...,4.0,0.0,106.0,0.0,28.0,0.105263,0.052632,0.000000,0.065000,2022-05-01
3,2022,5,9E,Endeavor Air Inc.,AEX,"Alexandria, LA: Alexandria International",88.0,14.0,7.26,0.76,...,585.0,35.0,125.0,0.0,202.0,0.159091,0.000000,0.054286,0.518571,2022-05-01
4,2022,5,9E,Endeavor Air Inc.,AGS,"Augusta, GA: Augusta Regional at Bush Field",181.0,19.0,13.84,0.00,...,662.0,0.0,87.0,0.0,59.0,0.104972,0.000000,0.000000,0.728421,2022-05-01


In [19]:

print(df.columns)


Index(['year', 'month', 'carrier', 'carrier_name', 'airport', 'airport_name',
       'arr_flights', 'arr_del15', 'carrier_ct', 'weather_ct', 'nas_ct',
       'security_ct', 'late_aircraft_ct', 'arr_cancelled', 'arr_diverted',
       'arr_delay', 'carrier_delay', 'weather_delay', 'nas_delay',
       'security_delay', 'late_aircraft_delay'],
      dtype='object')


In [32]:
df.shape



(318017, 26)

In [ ]:
df.describe()


,Age
count,98619.000000
mean,45.504021
std,25.929849
min,1.000000
25%,23.000000
50%,46.000000
75%,68.000000
max,90.000000
